# LAPD NIBRS data cleaning

## 1. Project vision and question

A street-lighting program manager at the City of Los Angeles Bureau of Street Lighting must decide which geographic areas with recurring nighttime reported offenses should be prioritized for on-site lighting assessments.

**Question:** Which LAPD reporting districts show persistent nighttime reported-offense patterns from January 1, 2025 forward and should receive further on-site lighting assessment?

I measure nighttime reported offenses, distinct cases, and months with nighttime reports. Persistence matters because one short spike should not automatically redirect public resources. This is a field-assessment aid, not a crime-prediction model or evidence that lighting caused an offense.

Each numbered section addresses an assignment item. The loading steps appear first so the row investigation uses the actual file. Short cells show the issue, apply a rule, and check the result. Only three review CSVs are exported.

## 3. Preserve the source

The supplied `LAPD_NIBRS_Offenses_Dataset.csv` was copied unchanged to `data/raw/lapd_nibrs_offenses_raw.csv`. Download date: **October 3, 2026, confirmed by Olga**; the source fingerprint identifies this file version. The filesystem date is not proof of extraction time.

The two standard-library tools below have narrow purposes: `Path` handles relative file locations, and `hashlib` verifies that the raw file's contents do not change. All analysis uses pandas. Jupyter supplies `display()` for showing ordinary tables.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data/raw').exists():
    ROOT = ROOT.parent

RAW = ROOT / 'data/raw/lapd_nibrs_offenses_raw.csv'
OUT = ROOT / 'data/cleaned'
OUT.mkdir(parents=True, exist_ok=True)
assert RAW.exists(), 'Place the original CSV in data/raw before running.'

with RAW.open('rb') as source_file:
    source_hash = hashlib.file_digest(source_file, 'sha256').hexdigest()
print('Source file:', RAW.name)
print('Source bytes:', RAW.stat().st_size)
print('Source fingerprint:', source_hash)

Source file: lapd_nibrs_offenses_raw.csv
Source bytes: 149127056
Source fingerprint: 8003ffd45a3e17938913b5794a5bad0a8078bfafec736591123005212614d1cf


In [2]:
# Read identifiers as strings so their leading zeros remain intact.
df_raw = pd.read_csv(RAW, dtype='string', keep_default_na=False)
df = df_raw.copy()
source_columns = df_raw.columns.tolist()
starting_rows = df_raw.shape[0]
df['source_row'] = df.index + 2  # Header is CSV row 1.

print('Original shape:', df_raw.shape)
if starting_rows == 2500:
    print('This file is the 2,500-row sample.')
else:
    print('This is not the 2,500-row sample; live-dataset completeness is unverified.')

Original shape: (489955, 31)
This is not the 2,500-row sample; live-dataset completeness is unverified.


## 4. Inspect columns and types

First check the exact column names. If a needed column is unavailable, the message below names it and stops dependent analysis. Empty strings are initially preserved so the before/after blank check is visible.

In [3]:
required = ['UniqueNIBRNo', 'CaseNo', 'Date OCC', 'Date Rptd', 'Time OCC',
            'AREA NAME', 'RPT Dist No', 'NIBR Code', 'NIBR Description',
            'Crime Against', 'Premise Desc', 'LOCATION', 'LAT', 'LON']
missing_columns = pd.Index(required).difference(df.columns)
print('Exact source columns:', source_columns)
print('Unavailable required columns:', missing_columns.tolist())
assert len(missing_columns) == 0, 'Required columns are unavailable; review the list above.'
df_raw.info()

Exact source columns: ['CaseNo', 'UniqueNIBRNo', 'Date Rptd', 'Date OCC', 'Time OCC', 'AREA', 'AREA NAME', 'RPT Dist No', 'TotalOffenseCount', 'Group', 'NIBR Code', 'NIBR Description', 'Crime Against', 'Premise Cd', 'Premise Desc', 'Weapon Used Cd', 'Weapon Desc', 'Status', 'Status Desc', 'TotalVictimCount', 'Victim Shot', 'Domestic Violence Crime', 'Hate Crime', 'Gang-related Crime', 'Transit-related Crime', 'Homeless-Victim Crime', 'Homeless-Suspect Crime', 'Homeless-Arrestee Crime', 'LOCATION', 'LAT', 'LON']
Unavailable required columns: []


<class 'pandas.DataFrame'>
RangeIndex: 489955 entries, 0 to 489954
Data columns (total 31 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   CaseNo                   489955 non-null  string
 1   UniqueNIBRNo             489955 non-null  string
 2   Date Rptd                489955 non-null  string
 3   Date OCC                 489955 non-null  string
 4   Time OCC                 489955 non-null  string
 5   AREA                     489955 non-null  string
 6   AREA NAME                489955 non-null  string
 7   RPT Dist No              489955 non-null  string
 8   TotalOffenseCount        489955 non-null  string
 9   Group                    489955 non-null  string
 10  NIBR Code                489955 non-null  string
 11  NIBR Description         489955 non-null  string
 12  Crime Against            489955 non-null  string
 13  Premise Cd               489955 non-null  string
 14  Premise Desc             489955

In [4]:
df_raw.head(5)

,CaseNo,UniqueNIBRNo,Date Rptd,Date OCC,Time OCC,AREA,AREA NAME,RPT Dist No,TotalOffenseCount,Group,...,Domestic Violence Crime,Hate Crime,Gang-related Crime,Transit-related Crime,Homeless-Victim Crime,Homeless-Suspect Crime,Homeless-Arrestee Crime,LOCATION,LAT,LON
0,25088246,25088246_13A_0,05/15/2025 02:23:02 PM,05/15/2025 02:21:00 PM,1421,05,Harbor,0558,3,A,...,No,No,Yes,No,No,No,No,400 N Palos Verdes St,33.746,-118.2817
1,25088246,25088246_290_2,05/15/2025 02:23:02 PM,05/15/2025 02:21:00 PM,1421,05,Harbor,0558,3,A,...,No,No,Yes,No,No,No,No,400 N Palos Verdes St,33.746,-118.2817
2,26170796,26170796_13B_0,08/31/2026 03:47:10 AM,08/31/2026 02:00:00 AM,0200,01,Central,0166,1,A,...,No,No,No,No,No,No,No,600 S San Pedro St,34.0423,-118.2453
3,26170797,26170797_13B_0,08/31/2026 02:55:11 AM,08/31/2026 01:30:21 AM,0130,15,N Hollywood,1503,1,A,...,No,No,No,No,No,No,No,12300 Sherman Way,34.2013,-118.4011
4,26170801,26170801_13B_0,08/30/2026 11:43:17 PM,08/30/2026 11:30:00 AM,1130,03,Southwest,0398,1,A,...,Yes,No,No,No,No,No,No,4100 S Hoover St,34.0094,-118.2871


In [5]:
# Count blank fields as missing in a separate inspection copy.
source_missing = df_raw.copy()
for column in source_columns:
    blank = source_missing[column].str.strip() == ''
    source_missing.loc[blank, column] = pd.NA

missing_before = source_missing.isna().sum()
missing_before.to_frame('missing_or_blank')

,missing_or_blank
CaseNo,0
UniqueNIBRNo,0
Date Rptd,7787
Date OCC,0
Time OCC,0
AREA,0
AREA NAME,0
RPT Dist No,0
TotalOffenseCount,0
Group,15480


In [6]:
df_raw[required].describe().T

,count,unique,top,freq
UniqueNIBRNo,489955,489955,25088246_13A_0,1
CaseNo,489955,440121,25219783,13
Date OCC,489955,210249,01/01/2025 12:00:00 AM,140
Date Rptd,489955,425978,,7787
Time OCC,489955,1440,0000,15333
AREA NAME,489955,21,Central,40582
RPT Dist No,489955,1183,0246,3203
NIBR Code,489955,59,13B,72672
NIBR Description,489955,2409,487(D)(1) - PC - F - Grand Theft Auto - GTA -...,30971
Crime Against,489955,5,Property,258973


## 2. Define one row

Compare offense IDs and case IDs separately. A repeated case may contain multiple offenses and is not automatically a duplicate. Unique offense IDs support treating rows as offense entries, but do not establish independent incidents or victims. I do not sum the case-level `TotalOffenseCount` across offense rows.

In [7]:
offense_ids = source_missing['UniqueNIBRNo']
case_ids = source_missing['CaseNo']

identifier_audit = pd.DataFrame(index=['UniqueNIBRNo', 'CaseNo'])
identifier_audit['total_rows'] = starting_rows
identifier_audit['distinct_nonmissing'] = [offense_ids.nunique(), case_ids.nunique()]
identifier_audit['missing'] = [offense_ids.isna().sum(), case_ids.isna().sum()]
identifier_audit['rows_in_repeated_groups'] = [
    (offense_ids.notna() & offense_ids.duplicated(keep=False)).sum(),
    (case_ids.notna() & case_ids.duplicated(keep=False)).sum()]
identifier_audit['repeats_beyond_first'] = [
    (offense_ids.notna() & offense_ids.duplicated()).sum(),
    (case_ids.notna() & case_ids.duplicated()).sum()]
identifier_audit = identifier_audit.rename_axis('identifier').reset_index()
identifier_audit

,identifier,total_rows,distinct_nonmissing,missing,rows_in_repeated_groups,repeats_beyond_first
0,UniqueNIBRNo,489955,489955,0,0,0
1,CaseNo,489955,440121,0,94224,49834


## 5. Standardize text

**Before:** inspect whitespace in descriptive labels. **Rule:** strip surrounding whitespace and join words with one space, only where a value changes. **After:** compare actual source and cleaned values. Capitalization, wording, and identifier zeros stay unchanged. This avoids splitting equivalent area or offense labels when planning assessments.

The short loop applies the same rule to five descriptive columns. `str.split().str.join(' ')` handles repeated whitespace without a regular expression.

In [8]:
text_columns = ['AREA NAME', 'NIBR Description', 'Crime Against', 'Premise Desc', 'LOCATION']
df[text_columns].head(5)

,AREA NAME,NIBR Description,Crime Against,Premise Desc,LOCATION
0,Harbor,245(A)(2) - PC - F - ADW w/Firearm On Person-A...,Person,Street/Parkway,400 N Palos Verdes St
1,Harbor,594(A)(2) - PC - F - Vandalism - Damage Proper...,Property,Street/Parkway,400 N Palos Verdes St
2,Central,243(A) - PC - M - Battery On Person - Simple -...,Person,Apartment/Condominium/Townhouse,600 S San Pedro St
3,N Hollywood,243(A) - PC - M - Battery On Person - Simple -...,Person,"Vehicle, Passenger/Truck",12300 Sherman Way
4,Southwest,243(E)(1) - PC - M - Intimate Partner Battery ...,Person,Victim's Residence,4100 S Hoover St


In [9]:
text_counts = pd.Series(0, index=text_columns, name='whitespace_changes')
for column in text_columns:
    cleaned_text = df[column].str.strip().str.split().str.join(' ')
    changed = df[column] != cleaned_text
    text_counts[column] = changed.sum()
    if changed.any():
        df[column + '_original'] = df[column]
        df[column] = cleaned_text

text_counts.to_frame()

,whitespace_changes
AREA NAME,489955
NIBR Description,82058
Crime Against,0
Premise Desc,0
LOCATION,0


In [10]:
# Show genuine examples. Brackets make surrounding spaces visible.
for column in text_columns:
    if text_counts[column] > 0:
        changed = df_raw[column] != df[column]
        examples = pd.DataFrame()
        examples['before'] = '[' + df_raw.loc[changed, column] + ']'
        examples['after'] = '[' + df.loc[changed, column] + ']'
        print(column)
        display(examples.head(5))

AREA NAME


,before,after
0,[Harbor ],[Harbor]
1,[Harbor ],[Harbor]
2,[Central ],[Central]
3,[N Hollywood ],[N Hollywood]
4,[Southwest ],[Southwest]


NIBR Description


,before,after
9,[487(D)(1) - PC - F - Grand Theft Auto - GTA ...,[487(D)(1) - PC - F - Grand Theft Auto - GTA -...
14,[487(D)(1) - PC - F - Grand Theft Auto - GTA ...,[487(D)(1) - PC - F - Grand Theft Auto - GTA -...
22,[273.5(A) - PC - F - IPV w/ Injury - Aggravate...,[273.5(A) - PC - F - IPV w/ Injury - Aggravate...
25,[487(D)(1) - PC - F - Grand Theft Auto - GTA ...,[487(D)(1) - PC - F - Grand Theft Auto - GTA -...
26,[484(A) - PC - M - Petty Theft - Theft From B...,[484(A) - PC - M - Petty Theft - Theft From Bu...


## 6. Investigate duplicates

**Before:** count exact rows, repeated offense IDs, and repeated cases separately; inspect examples. **Rule:** remove only later copies whose original source fields are all identical. **After:** count retained rows and repeat groups. Non-identical repeated offense IDs need investigation; multiple offenses in the same case remain legitimate evidence.

In [11]:
exact_duplicate = df_raw.duplicated(keep='first')
duplicate_count = exact_duplicate.sum()
print('Exact duplicate rows beyond the first:', duplicate_count)
df_raw.loc[df_raw.duplicated(keep=False)].head(6)

Exact duplicate rows beyond the first: 0


,CaseNo,UniqueNIBRNo,Date Rptd,Date OCC,Time OCC,AREA,AREA NAME,RPT Dist No,TotalOffenseCount,Group,...,Domestic Violence Crime,Hate Crime,Gang-related Crime,Transit-related Crime,Homeless-Victim Crime,Homeless-Suspect Crime,Homeless-Arrestee Crime,LOCATION,LAT,LON


In [12]:
repeated_offense = offense_ids.notna() & offense_ids.duplicated(keep=False)
repeated_case = case_ids.notna() & case_ids.duplicated(keep=False)
print('Rows in repeated offense-ID groups:', repeated_offense.sum())
print('Rows in repeated case groups:', repeated_case.sum())
display(df_raw.loc[repeated_offense, ['UniqueNIBRNo', 'CaseNo', 'NIBR Code']].head(6))
display(df_raw.loc[repeated_case, ['UniqueNIBRNo', 'CaseNo', 'NIBR Code']].head(6))

Rows in repeated offense-ID groups: 0
Rows in repeated case groups: 94224


,UniqueNIBRNo,CaseNo,NIBR Code


,UniqueNIBRNo,CaseNo,NIBR Code
0,25088246_13A_0,25088246,13A
1,25088246_290_2,25088246,290
6,26170803_23F_0,26170803,23F
7,26170803_290_1,26170803,290
11,26170815_23F_0,26170815,23F
12,26170815_290_1,26170815,290


In [13]:
df = df.loc[~exact_duplicate].copy()
print('Rows before:', starting_rows)
print('Exact copies removed:', duplicate_count)
print('Rows after:', df.shape[0])

Rows before: 489955
Exact copies removed: 0
Rows after: 489955


## 7. Handle missing values

**Before:** use the missingness counts from section 4. **Rule:** convert empty/whitespace-only source fields to missing, and do not guess missing premises, times, identifiers, locations, or coordinates. **After:** show missingness and explicit flags. Missing values can affect which districts appear to need attention, so they remain visible rather than being filled in.

In [14]:
for column in source_columns:
    blank = df[column].str.strip() == ''
    df.loc[blank, column] = pd.NA

missing_after_blanks = df[source_columns].isna().sum()
missing_after_blanks.to_frame('missing_after_blank_rule')

,missing_after_blank_rule
CaseNo,0
UniqueNIBRNo,0
Date Rptd,7787
Date OCC,0
Time OCC,0
AREA,0
AREA NAME,0
RPT Dist No,0
TotalOffenseCount,0
Group,15480


In [15]:
df['flag_missing_premise'] = df['Premise Desc'].isna()
df['flag_missing_location'] = df['LOCATION'].isna()
df['flag_missing_identifier'] = df[['UniqueNIBRNo', 'CaseNo']].isna().any(axis=1)
df['flag_repeated_UniqueNIBRNo'] = df['UniqueNIBRNo'].notna() & df['UniqueNIBRNo'].duplicated(keep=False)
df['flag_repeated_CaseNo'] = df['CaseNo'].notna() & df['CaseNo'].duplicated(keep=False)
df[['flag_missing_premise', 'flag_missing_location', 'flag_missing_identifier',
    'flag_repeated_UniqueNIBRNo', 'flag_repeated_CaseNo']].sum()

flag_missing_premise            109
flag_missing_location             0
flag_missing_identifier           0
flag_repeated_UniqueNIBRNo        0
flag_repeated_CaseNo          94224
dtype: int64

## 8. Check suspicious values

### Dates: inspect, convert, and count failures

Original date strings are retained. Conversion failures count only originally nonmissing values. Missing report dates stay missing. Date conversion adds columns but does not remove rows. The format below matches the observed source strings.

In [16]:
df[['Date OCC', 'Date Rptd']].head(5)

,Date OCC,Date Rptd
0,05/15/2025 02:21:00 PM,05/15/2025 02:23:02 PM
1,05/15/2025 02:21:00 PM,05/15/2025 02:23:02 PM
2,08/31/2026 02:00:00 AM,08/31/2026 03:47:10 AM
3,08/31/2026 01:30:21 AM,08/31/2026 02:55:11 AM
4,08/30/2026 11:30:00 AM,08/30/2026 11:43:17 PM


In [17]:
rows_before_conversion = df.shape[0]
df['Date OCC_original'] = df['Date OCC']
df['Date OCC'] = pd.to_datetime(df['Date OCC'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')
df['flag_invalid_Date_OCC'] = df['Date OCC_original'].notna() & df['Date OCC'].isna()

df['Date Rptd_original'] = df['Date Rptd']
df['Date Rptd'] = pd.to_datetime(df['Date Rptd'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')
df['flag_invalid_Date_Rptd'] = df['Date Rptd_original'].notna() & df['Date Rptd'].isna()

date_failures = pd.Series(index=['Date OCC', 'Date Rptd'], dtype='int64')
date_failures['Date OCC'] = df['flag_invalid_Date_OCC'].sum()
date_failures['Date Rptd'] = df['flag_invalid_Date_Rptd'].sum()
print('Rows before and after date conversion:', rows_before_conversion, df.shape[0])
date_failures.to_frame('nonmissing_conversion_failures')

Rows before and after date conversion: 489955 489955


,nonmissing_conversion_failures
Date OCC,0.0
Date Rptd,0.0


### Dates: flag unusual values without deleting them

“Future” uses the real extraction date if entered, otherwise the execution date as a clearly labeled proxy. Comparing calendar dates avoids treating an earlier report time on the same day as a date reversal. Before 2000 is a screening definition of “very old,” not proof of error. Pre-2024, all 2024, and January 1–March 6, 2024 records are flagged separately because the March 7 transition limits comparability.

In [18]:
DOWNLOAD_DATE = '2026-10-03'  # Download date confirmed by Olga; used as the extraction-context reference.
RUN_DATE = pd.Timestamp.now(tz='America/Los_Angeles').tz_localize(None).normalize()
if DOWNLOAD_DATE is None:
    DATE_CONTEXT = RUN_DATE
    print('Date reference is the execution date; extraction date is unknown.')
else:
    DATE_CONTEXT = pd.Timestamp(DOWNLOAD_DATE)
print('Date reference:', DATE_CONTEXT.date())

occurrence_day = df['Date OCC'].dt.normalize()
report_day = df['Date Rptd'].dt.normalize()
df['flag_occ_after_report'] = occurrence_day > report_day
df['flag_future_date'] = (occurrence_day > DATE_CONTEXT) | (report_day > DATE_CONTEXT)
df['flag_very_old_occurrence'] = df['Date OCC'] < '2000-01-01'
df['flag_pre2024'] = df['Date OCC'] < '2024-01-01'
df['flag_transition_2024'] = df['Date OCC'].dt.year == 2024
df['flag_before_transition_2024'] = df['Date OCC'].between('2024-01-01', '2024-03-06 23:59:59')

Date reference: 2026-10-03


In [19]:
date_flags = ['flag_invalid_Date_OCC', 'flag_invalid_Date_Rptd', 'flag_occ_after_report',
              'flag_future_date', 'flag_very_old_occurrence', 'flag_pre2024',
              'flag_transition_2024', 'flag_before_transition_2024']
display(df[date_flags].sum().to_frame('flagged_rows'))
unusual_date = df[date_flags].any(axis=1)
df.loc[unusual_date, ['source_row', 'Date OCC_original', 'Date Rptd_original']].head(5)

,flagged_rows
flag_invalid_Date_OCC,0
flag_invalid_Date_Rptd,0
flag_occ_after_report,4
flag_future_date,0
flag_very_old_occurrence,165
flag_pre2024,6824
flag_transition_2024,103040
flag_before_transition_2024,1351


,source_row,Date OCC_original,Date Rptd_original
150,152,08/08/2024 12:00:00 AM,09/12/2026 10:09:18 AM
221,223,01/01/2021 11:15:00 AM,09/01/2026 11:21:33 AM
292,294,07/01/2017 11:00:56 AM,<NA>
302,304,01/01/2018 12:00:00 AM,09/01/2026 10:24:41 PM
351,353,08/20/2023 01:00:00 AM,<NA>


### Times: inspect original values

Only digit strings of one to four characters are candidates for HHMM. Padding preserves leading zeros; hours must be 00–23 and minutes 00–59. Invalid or missing times stay in the master, outside the percentage denominator. `0000` is retained and flagged because the file cannot distinguish genuine midnight from uncertain precision.

In [20]:
print('Time storage:', df_raw['Time OCC'].dtype)
df_raw['Time OCC'].value_counts(dropna=False).head(10)

Time storage: string


Time OCC
0000    15333
1200    13314
1800    10103
1700    10017
2000     9708
1900     9056
2200     8992
2100     8719
1600     8419
1500     8240
Name: count, dtype: Int64

In [21]:
df['Time OCC_original'] = df['Time OCC']
time_text = df['Time OCC'].str.strip()
digits = time_text.str.isdecimal() & time_text.str.len().between(1, 4)
digits = digits.fillna(False)
padded = time_text.where(digits).str.zfill(4)
hours = pd.to_numeric(padded.str[:2], errors='coerce')
minutes = pd.to_numeric(padded.str[2:], errors='coerce')

df['flag_valid_time'] = digits & hours.between(0, 23) & minutes.between(0, 59)
df['flag_valid_time'] = df['flag_valid_time'].fillna(False)
df['flag_missing_time'] = df['Time OCC'].isna()
df['flag_invalid_time'] = ~df['flag_valid_time'] & ~df['flag_missing_time']
df['time_hhmm'] = padded.where(df['flag_valid_time'])
df['occurrence_hour'] = hours.where(df['flag_valid_time']).astype('Int64')
df['flag_0000_ambiguous'] = df['time_hhmm'].eq('0000').fillna(False)

In [22]:
time_flags = ['flag_valid_time', 'flag_missing_time', 'flag_invalid_time', 'flag_0000_ambiguous']
display(df[time_flags].sum().to_frame('records'))
df[['Time OCC_original', 'time_hhmm', 'occurrence_hour', 'flag_0000_ambiguous']].head(5)

,records
flag_valid_time,489955
flag_missing_time,0
flag_invalid_time,0
flag_0000_ambiguous,15333


,Time OCC_original,time_hhmm,occurrence_hour,flag_0000_ambiguous
0,1421,1421,14,False
1,1421,1421,14,False
2,0200,0200,2,False
3,0130,0130,1,False
4,1130,1130,11,False


### Coordinates: inspect, convert, and flag

Non-numeric values become missing numeric values with original strings preserved. The broad Los Angeles-area box is latitude 33.5–34.9 and longitude -119.0–-117.5. It is a screening rule, not a city boundary. Zero, missing, and outlying coordinates are retained for review before any future spatial matching.

In [23]:
display(df[['LAT', 'LON']].head(5))
df[['LAT', 'LON']].isna().sum()

,LAT,LON
0,33.746,-118.2817
1,33.746,-118.2817
2,34.0423,-118.2453
3,34.2013,-118.4011
4,34.0094,-118.2871


LAT    131
LON    131
dtype: int64

In [24]:
df['LAT_original'] = df['LAT']
df['LAT'] = pd.to_numeric(df['LAT'], errors='coerce')
df['LON_original'] = df['LON']
df['LON'] = pd.to_numeric(df['LON'], errors='coerce')

df['flag_missing_coordinates'] = df[['LAT_original', 'LON_original']].isna().any(axis=1)
bad_latitude = df['LAT_original'].notna() & df['LAT'].isna()
bad_longitude = df['LON_original'].notna() & df['LON'].isna()
df['flag_invalid_coordinates'] = bad_latitude | bad_longitude
df['flag_zero_coordinates'] = (df['LAT'].eq(0) | df['LON'].eq(0)).fillna(False)
both_coordinates = df[['LAT', 'LON']].notna().all(axis=1)
inside_box = df['LAT'].between(33.5, 34.9) & df['LON'].between(-119.0, -117.5)
df['flag_outside_la_box'] = (both_coordinates & ~inside_box).fillna(False)

In [25]:
coordinate_flags = ['flag_missing_coordinates', 'flag_invalid_coordinates',
                    'flag_zero_coordinates', 'flag_outside_la_box']
display(df[coordinate_flags].sum().to_frame('flagged_rows'))
suspicious_coordinates = df[coordinate_flags].any(axis=1)
df.loc[suspicious_coordinates, ['source_row', 'LAT_original', 'LON_original', 'LAT', 'LON']].head(5)

,flagged_rows
flag_missing_coordinates,131
flag_invalid_coordinates,0
flag_zero_coordinates,550
flag_outside_la_box,551


,source_row,LAT_original,LON_original,LAT,LON
2549,2551,0,0,0.0,0.0
3748,3750,0,0,0.0,0.0
4158,4160,0,0,0.0,0.0
9483,9485,0,0,0.0,0.0
11403,11405,<NA>,<NA>,<NA>,<NA>


### Missingness before and after all conversions

This table includes every source column, so changes caused by duplicate removal, blank standardization, or conversion remain visible. No missing value was filled with a guessed value.

In [26]:
missing_audit = pd.DataFrame()
missing_audit['raw_missing_or_blank'] = missing_before
missing_audit['after_dedup_and_blank_standardization'] = missing_after_blanks
missing_audit['after_conversions'] = df[source_columns].isna().sum()
missing_audit

,raw_missing_or_blank,after_dedup_and_blank_standardization,after_conversions
CaseNo,0,0,0
UniqueNIBRNo,0,0,0
Date Rptd,7787,7787,7787
Date OCC,0,0,0
Time OCC,0,0,0
AREA,0,0,0
AREA NAME,0,0,0
RPT Dist No,0,0,0
TotalOffenseCount,0,0,0
Group,15480,15480,15480


## 9. Verify merges

**Not applicable for the current cleaning stage.** No streetlight file is merged. A future spatial join requires explicit approval and checks of coordinate systems, matching distances, unmatched records, and row multiplication. Offense locations alone cannot establish lighting quality.

## 10. Reconcile changes

Only exact duplicate source rows can be removed. Date conversion and uncertainty flags remove no rows. Backup columns and derived fields increase the column count, not the row count.

In [27]:
reconciliation = pd.DataFrame([
    ['Starting rows', starting_rows],
    ['Exact duplicates removed', duplicate_count],
    ['Other rules removed', rows_before_conversion - df.shape[0]],
    ['Final rows', df.shape[0]],
    ['Net difference', df.shape[0] - starting_rows]
], columns=['measure', 'count'])
display(reconciliation)
print('Columns before and after:', len(source_columns), df.shape[1])
assert starting_rows - duplicate_count == df.shape[0], 'Row accounting does not balance.'
assert rows_before_conversion == df.shape[0], 'Date conversion changed the row count.'

,measure,count
0,Starting rows,489955
1,Exact duplicates removed,0
2,Other rules removed,0
3,Final rows,489955
4,Net difference,0


Columns before and after: 31 62


## 11. Check calculations

Nighttime is **20:00 through 05:59**, our analysis rule rather than a source category. I show an inclusive count, but exclude ambiguous `0000` from the main nighttime count and percentage denominator. Invalid and missing times are also excluded. The counts are offense entries, not people or exposure-adjusted risk.

In [28]:
night_hours = (df['occurrence_hour'] >= 20) | (df['occurrence_hour'] < 6)
df['nighttime_including_0000'] = (df['flag_valid_time'] & night_hours).fillna(False)
df['nighttime_unambiguous'] = df['nighttime_including_0000'] & ~df['flag_0000_ambiguous']
df['analysis_post2024'] = df['Date OCC'] >= '2025-01-01'
df['analysis_date_eligible'] = df['analysis_post2024'] & (occurrence_day <= DATE_CONTEXT)
df['occurrence_month'] = df['Date OCC'].dt.to_period('M').astype('string')

In [29]:
eligible_time = df['flag_valid_time'] & ~df['flag_0000_ambiguous']
denominator = eligible_time.sum()
night_count = df['nighttime_unambiguous'].sum()
nighttime_percent = float('nan')
if denominator > 0:
    nighttime_percent = 100 * night_count / denominator

metrics = pd.Series(index=[
    'raw_rows', 'cleaned_rows', 'valid_time_records', 'invalid_nonmissing_time_records',
    'missing_time_records', 'ambiguous_0000_records', 'nighttime_including_0000',
    'nighttime_unambiguous', 'percentage_denominator', 'denominator_excluded',
    'nighttime_percent', 'post2024_records'], dtype='float64')
metrics['raw_rows'] = starting_rows
metrics['cleaned_rows'] = df.shape[0]
metrics['valid_time_records'] = df['flag_valid_time'].sum()
metrics['invalid_nonmissing_time_records'] = df['flag_invalid_time'].sum()
metrics['missing_time_records'] = df['flag_missing_time'].sum()
metrics['ambiguous_0000_records'] = df['flag_0000_ambiguous'].sum()
metrics['nighttime_including_0000'] = df['nighttime_including_0000'].sum()
metrics['nighttime_unambiguous'] = night_count
metrics['percentage_denominator'] = denominator
metrics['denominator_excluded'] = df.shape[0] - denominator
metrics['nighttime_percent'] = nighttime_percent
metrics['post2024_records'] = df['analysis_post2024'].sum()
metrics.to_frame('value')

,value
raw_rows,489955.000000
cleaned_rows,489955.000000
valid_time_records,489955.000000
invalid_nonmissing_time_records,0.000000
missing_time_records,0.000000
ambiguous_0000_records,15333.000000
nighttime_including_0000,178762.000000
nighttime_unambiguous,163429.000000
percentage_denominator,474622.000000
denominator_excluded,15333.000000


In [30]:
excluded_times = df['flag_invalid_time'].sum() + df['flag_missing_time'].sum() + df['flag_0000_ambiguous'].sum()
assert df.shape[0] - denominator == excluded_times, 'The time denominator does not reconcile.'
print('Excluded invalid times:', df['flag_invalid_time'].sum())
print('Excluded missing times:', df['flag_missing_time'].sum())
print('Excluded ambiguous zero times:', df['flag_0000_ambiguous'].sum())

Excluded invalid times: 0
Excluded missing times: 0
Excluded ambiguous zero times: 15333


### Districts and persistence across months

Earlier records remain in the master. The focus uses 2025-forward occurrence dates that are not future relative to our reference date, and valid unambiguous nighttime records. `dropna=False` keeps missing group keys visible. Distinct-case counts exclude missing case IDs, so those are reported separately.

Active-month share divides by a common observed calendar window; this is not verified coverage and may include partial months. Review the monthly pattern rather than treating one large total as persistent need. The tables below remain in the notebook, not separate CSVs.

In [31]:
analysis = df.loc[df['analysis_date_eligible']].copy()
night = analysis.loc[analysis['nighttime_unambiguous']].copy()
print('Missing district keys in master:', df['RPT Dist No'].isna().sum())
print('Missing district keys in focus:', analysis['RPT Dist No'].isna().sum())
print('Missing case IDs in nighttime focus:', night['CaseNo'].isna().sum())
print('Focus date range:', analysis['Date OCC'].min(), 'through', analysis['Date OCC'].max())

Missing district keys in master: 0
Missing district keys in focus: 0
Missing case IDs in nighttime focus: 0
Focus date range: 2025-01-01 00:00:00 through 2026-09-19 00:00:00


In [32]:
district = night.groupby('RPT Dist No', dropna=False).agg(
    nighttime_offenses=('source_row', 'size'),
    distinct_cases=('CaseNo', 'nunique'),
    active_months=('occurrence_month', 'nunique'))
observed_months = pd.period_range(analysis['Date OCC'].min(), analysis['Date OCC'].max(), freq='M')
district['common_observed_calendar_months'] = len(observed_months)
district['active_month_share'] = district['active_months'] / len(observed_months)
district = district.sort_values(['active_months', 'nighttime_offenses'], ascending=False)
print('Complete district table shape:', district.shape)
print('Top 10 districts by active months, then nighttime offense count:')
district.head(10)

Complete district table shape: (1158, 5)
Top 10 districts by active months, then nighttime offense count:


,nighttime_offenses,distinct_cases,active_months,common_observed_calendar_months,active_month_share
RPT Dist No,,,,,
0182,870,805,21,21,1.0
1494,829,772,21,21,1.0
0636,790,724,21,21,1.0
0646,743,684,21,21,1.0
0162,686,647,21,21,1.0
0904,673,604,21,21,1.0
0111,631,560,21,21,1.0
0666,601,547,21,21,1.0
0645,566,517,21,21,1.0


In [33]:
monthly = night.groupby(['RPT Dist No', 'occurrence_month'], dropna=False).agg(
    nighttime_offenses=('source_row', 'size'),
    distinct_cases=('CaseNo', 'nunique'))
print('Complete district-month table shape:', monthly.shape)
print('Monthly example: first 10 district-month groups, not a ranking.')
monthly.head(10)

Complete district-month table shape: (21083, 2)
Monthly example: first 10 district-month groups, not a ranking.


nighttime_offenses  distinct_cases
RPT Dist No occurrence_month                                    
01          2025-01                            3               3
            2025-02                            4               4
            2025-03                            3               3
0101        2025-01                           10               9
            2025-02                            9               8
            2025-03                           10               8
            2025-04                           14              12
            2025-05                           10              10
            2025-06                            5               5
            2025-07                           15              14

In [34]:
quality = df.groupby('RPT Dist No', dropna=False).agg(
    offense_rows=('source_row', 'size'),
    ambiguous_0000=('flag_0000_ambiguous', 'sum'),
    invalid_time=('flag_invalid_time', 'sum'),
    missing_time=('flag_missing_time', 'sum'),
    future_date_flags=('flag_future_date', 'sum'),
    repeated_offense_id=('flag_repeated_UniqueNIBRNo', 'sum'))
print('Complete district-quality table shape:', quality.shape)
print('Quality example: first 10 reporting districts.')
quality.head(10)

Complete district-quality table shape: (1183, 6)
Quality example: first 10 reporting districts.


,offense_rows,ambiguous_0000,invalid_time,missing_time,future_date_flags,repeated_offense_id
RPT Dist No,,,,,,
01,35,1,0,0,0,0
0101,556,27,0,0,0,0
0105,169,3,0,0,0,0
0109,23,0,0,0,0,0
0111,2092,73,0,0,0,0
0112,191,6,0,0,0,0
0116,1,0,0,0,0,0
0118,1601,27,0,0,0,0
0119,1528,41,0,0,0,0


### Complete tables remain available

The previews above limit displayed rows. The complete `district`, `monthly`, and `quality` DataFrames remain in memory; their shapes are reported above. No underlying calculation or table has been changed.

## 12. Verify five individual records

### Select source records and state expectations first

The following selectors examine original source fields, not the cleaned flags being tested. The expected True/False values state the rule for each chosen type before actual results are retrieved. Five distinct retained rows are chosen. The supplied file contains each required type; the assertion clearly stops if a future file lacks one so a documented substitute can be selected.

For this independent time check, dividing numeric HHMM by 100 gives the hour, and the remainder gives the minute. This is separate from the string-slicing cleaning rule.

In [35]:
original_time = pd.to_numeric(df_raw['Time OCC'], errors='coerce')
original_hours = original_time // 100
original_minutes = original_time % 100
original_valid = original_hours.between(0, 23) & original_minutes.between(0, 59)
original_zero = original_time == 0
original_dates = pd.to_datetime(df_raw['Date OCC'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')
original_night = (original_hours >= 20) | (original_hours < 6)
case_frequency = df_raw['CaseNo'].value_counts()
multiple_case_rows = df_raw['CaseNo'].map(case_frequency) > 1
retained = ~exact_duplicate

typical_rows = df_raw.loc[retained & original_valid & original_night & ~original_zero]
zero_rows = df_raw.loc[retained & original_zero]
premise_rows = df_raw.loc[retained & (df_raw['Premise Desc'].str.strip() == '')]
historical_rows = df_raw.loc[retained & (original_dates < '2025-01-01')]
case_rows = df_raw.loc[retained & multiple_case_rows]
print('Available typical, zero-time, missing-premise, historical, repeated-case examples:')
print(len(typical_rows), len(zero_rows), len(premise_rows), len(historical_rows), len(case_rows))

Available typical, zero-time, missing-premise, historical, repeated-case examples:
163429 15333 109 109864 94224


In [36]:
chosen = []
for candidates in [typical_rows, zero_rows, premise_rows, historical_rows, case_rows]:
    available = candidates.drop(index=chosen, errors='ignore')
    assert len(available) > 0, 'A record type is unavailable; select and document a different difficult case.'
    chosen.append(available.index[0])

expected_checks = df_raw.loc[chosen, ['UniqueNIBRNo']].copy()
expected_checks.insert(0, 'source_row', expected_checks.index + 2)
expected_checks['selection_reason'] = ['Typical valid nighttime', 'Ambiguous midnight',
    'Missing premise', 'Historical occurrence', 'Multiple offenses in a case']
expected_checks['field_checked'] = ['nighttime_unambiguous', 'flag_0000_ambiguous',
    'flag_missing_premise', 'analysis_post2024', 'flag_repeated_CaseNo']
expected_checks['original_value'] = [df_raw.loc[chosen[0], 'Time OCC'], df_raw.loc[chosen[1], 'Time OCC'],
    df_raw.loc[chosen[2], 'Premise Desc'], df_raw.loc[chosen[3], 'Date OCC'], df_raw.loc[chosen[4], 'CaseNo']]
expected_checks['expected_result'] = [True, True, True, False, True]
expected_checks['reason'] = [
    'Original HHMM lies in 20:00–05:59 and is not all zeros.',
    'All-zero original time remains present and gets an ambiguity flag.',
    'An empty original premise cannot be inferred from the offense and remains missing.',
    'The original occurrence precedes January 1, 2025; retain it outside the focus flag.',
    'Other original offense rows share this case; case repetition is preserved.']
expected_checks = expected_checks.reset_index(drop=True)
expected_checks

,source_row,UniqueNIBRNo,selection_reason,field_checked,original_value,expected_result,reason
0,4,26170796_13B_0,Typical valid nighttime,nighttime_unambiguous,0200,True,Original HHMM lies in 20:00–05:59 and is not a...
1,55,26172245_220_0,Ambiguous midnight,flag_0000_ambiguous,0000,True,All-zero original time remains present and get...
2,630,971726272__1,Missing premise,flag_missing_premise,,True,An empty original premise cannot be inferred f...
3,152,26179469_270_0,Historical occurrence,analysis_post2024,08/08/2024 12:00:00 AM,False,"The original occurrence precedes January 1, 20..."
4,2,25088246_13A_0,Multiple offenses in a case,flag_repeated_CaseNo,25088246,True,Other original offense rows share this case; c...


### Retrieve actual cleaned results and calculate matches

Review the expectations above before this cell. False matches must be investigated, not relabeled as passing.

In [37]:
checks = expected_checks.copy()
checks['actual_cleaned_result'] = [
    df.loc[chosen[0], 'nighttime_unambiguous'],
    df.loc[chosen[1], 'flag_0000_ambiguous'],
    df.loc[chosen[2], 'flag_missing_premise'],
    df.loc[chosen[3], 'analysis_post2024'],
    df.loc[chosen[4], 'flag_repeated_CaseNo']]
checks['match'] = checks['expected_result'] == checks['actual_cleaned_result']
display(checks)
checks.to_csv(OUT / 'five_record_checks.csv', index=False)

,source_row,UniqueNIBRNo,selection_reason,field_checked,original_value,expected_result,reason,actual_cleaned_result,match
0,4,26170796_13B_0,Typical valid nighttime,nighttime_unambiguous,0200,True,Original HHMM lies in 20:00–05:59 and is not a...,True,True
1,55,26172245_220_0,Ambiguous midnight,flag_0000_ambiguous,0000,True,All-zero original time remains present and get...,True,True
2,630,971726272__1,Missing premise,flag_missing_premise,,True,An empty original premise cannot be inferred f...,True,True
3,152,26179469_270_0,Historical occurrence,analysis_post2024,08/08/2024 12:00:00 AM,False,"The original occurrence precedes January 1, 20...",False,True
4,2,25088246_13A_0,Multiple offenses in a case,flag_repeated_CaseNo,25088246,True,Other original offense rows share this case; c...,True,True


## Independent check of a Codex suggestion

**Suggestion:** Codex suggested that `Time OCC = 0000` should not automatically be converted to missing because it could represent either a genuine midnight occurrence or uncertain time precision.

The source time storage was inspected in section 8. Now count literal zero times and compare dates and offense descriptions. These patterns cannot establish LAPD's intended meaning.

In [38]:
print('Source time storage:', df_raw['Time OCC'].dtype)
print('Literal 0000 source records:', (df_raw['Time OCC'] == '0000').sum())
print('Retained ambiguous zero times:', df['flag_0000_ambiguous'].sum())
df.loc[df['flag_0000_ambiguous'], ['UniqueNIBRNo', 'Time OCC_original', 'time_hhmm',
    'Date OCC_original', 'Date Rptd_original', 'NIBR Code', 'NIBR Description']].head(8)

Source time storage: string
Literal 0000 source records: 15333
Retained ambiguous zero times: 15333


,UniqueNIBRNo,Time OCC_original,time_hhmm,Date OCC_original,Date Rptd_original,NIBR Code,NIBR Description
53,26172245_220_0,0000,0000,08/31/2026 12:00:00 AM,09/01/2026 08:22:37 PM,220,459 - PC - F - Burglary - Residential - 220
99,26173218_13C_0,0000,0000,04/01/2026 12:00:00 AM,09/03/2026 06:30:31 AM,13C,422(A) - PC - F - Criminal Threats - 13C
136,26174735_290_0,0000,0000,09/05/2026 12:00:00 AM,09/05/2026 11:38:39 AM,290,594(B)(1) - PC - F - Vandalism ($400 Or More) ...
150,26179469_270_0,0000,0000,08/08/2024 12:00:00 AM,09/12/2026 10:09:18 AM,270,503 - PC - F - Embezzlement ($950 or more) - F...
250,26170835_13B_0,0000,0000,08/31/2026 12:00:00 AM,08/31/2026 08:01:47 AM,13B,273D(A) - PC - M - Child Abuse - Simple - 13B
253,26170838_220_0,0000,0000,07/31/2026 12:00:00 AM,08/31/2026 07:29:15 AM,220,459 - PC - F - Burglary - Residential - 220
302,26172275_11D_0,0000,0000,01/01/2018 12:00:00 AM,09/01/2026 10:24:41 PM,11D,288(C)(1) - PC - F - Lewd And Lascivious Acts ...
346,26173246_13B_0,0000,0000,09/02/2026 12:00:00 AM,09/03/2026 10:19:30 AM,13B,242 - PC - M - Battery On Person - Simple - 13B


### Look for a definition in existing project documents

This short search reads the existing project documents rather than treating generated README/notebook text as independent evidence. A search hit must be read in context. No definition of `0000` is established merely by finding that string.

In [39]:
for filename in ['project-start.md', 'members.md']:
    document = ROOT / filename
    if document.exists():
        lines = pd.Series(document.read_text(encoding='utf-8').splitlines())
        words = lines.str.lower()
        hit = words.str.contains('0000', regex=False) | words.str.contains('midnight', regex=False)
        hit = hit | words.str.contains('time occ', regex=False) | words.str.contains('unknown time', regex=False)
        print('Searched:', filename, '| matching lines:', hit.sum())
        display(lines.loc[hit])

Searched: project-start.md | matching lines: 0


Series([], dtype: str)

Searched: members.md | matching lines: 0


Series([], dtype: str)

I did not accept Codex's suggestion automatically. I checked the time storage, counted zero-time records, compared their date and offense fields, and looked for a definition in the project documents. The available evidence does not distinguish genuine midnight from uncertain precision, so I preserved these values with a flag. I excluded them from the main nighttime percentage and showed the inclusive count separately. I want the program manager to see where uncertainty enters a ranking, especially when it may affect which communities receive attention and resources.

## Cleaning decision log

The rows below record the actual rules and calculated affected counts. Each small cell adds a few decisions to an ordinary DataFrame. Counts can overlap: flags are not row deletions.

In [40]:
log = pd.DataFrame(columns=['rule', 'project_specific_reason', 'verification_check',
    'affected_records', 'final_treatment', 'remaining_limitation'])
blank_rows = source_missing.loc[~exact_duplicate].isna().any(axis=1).sum()

In [41]:
log.loc[len(log)] = [
    'Preserve source and identifiers',
    'Keep assessment priorities traceable to original offense entries.',
    'SHA-256 before and after; strings preserve leading zeros.',
    starting_rows,
    'Source unchanged; work on a copy.',
    'Live-portal completeness unverified.']

log.loc[len(log)] = [
    'Whitespace: ' + 'AREA NAME',
    'Avoid splitting equivalent location or offense labels in assessment summaries.',
    'Compare original and normalized strings.',
    text_counts['AREA NAME'],
    'Trim/collapse whitespace only when changed; back up originals.',
    'Wording and capitalization are not reconciled.']

log.loc[len(log)] = [
    'Whitespace: ' + 'NIBR Description',
    'Avoid splitting equivalent location or offense labels in assessment summaries.',
    'Compare original and normalized strings.',
    text_counts['NIBR Description'],
    'Trim/collapse whitespace only when changed; back up originals.',
    'Wording and capitalization are not reconciled.']

In [42]:
log.loc[len(log)] = [
    'Whitespace: ' + 'Crime Against',
    'Avoid splitting equivalent location or offense labels in assessment summaries.',
    'Compare original and normalized strings.',
    text_counts['Crime Against'],
    'Trim/collapse whitespace only when changed; back up originals.',
    'Wording and capitalization are not reconciled.']

log.loc[len(log)] = [
    'Whitespace: ' + 'Premise Desc',
    'Avoid splitting equivalent location or offense labels in assessment summaries.',
    'Compare original and normalized strings.',
    text_counts['Premise Desc'],
    'Trim/collapse whitespace only when changed; back up originals.',
    'Wording and capitalization are not reconciled.']

log.loc[len(log)] = [
    'Whitespace: ' + 'LOCATION',
    'Avoid splitting equivalent location or offense labels in assessment summaries.',
    'Compare original and normalized strings.',
    text_counts['LOCATION'],
    'Trim/collapse whitespace only when changed; back up originals.',
    'Wording and capitalization are not reconciled.']

In [43]:
log.loc[len(log)] = [
    'Blank strings to missing',
    'Do not invent premise or location evidence for field assessments.',
    'Before/after missingness audit.',
    blank_rows,
    'Preserve missingness; add flags.',
    'Missing information may vary by district.']

log.loc[len(log)] = [
    'Exact source duplicates',
    'Avoid double-counting identical supplied offense entries.',
    'All original columns match.',
    duplicate_count,
    'Keep first exact copy only.',
    'Distinct rows with repeated IDs remain unresolved.']

log.loc[len(log)] = [
    'Repeated case IDs',
    'Multiple offenses in one case must not disappear from assessment evidence.',
    'Compare offense IDs and case IDs.',
    df['flag_repeated_CaseNo'].sum(),
    'Retain; report distinct cases separately.',
    'Offense counts and case counts differ.']

In [44]:
log.loc[len(log)] = [
    'Repeated offense IDs',
    'Avoid silently resolving conflicting offense entries.',
    'Identifier audit and sample rows.',
    df['flag_repeated_UniqueNIBRNo'].sum(),
    'Flag and retain.',
    'Nonunique IDs can affect interpretation of offense counts.']

log.loc[len(log)] = [
    'Date conversion: ' + 'Date OCC',
    'Month persistence requires usable dates.',
    'Nonmissing input converted to NaT.',
    date_failures['Date OCC'],
    'Preserve backup; flag failures; exclude unusable dates from dated analysis.',
    'Cannot recover invalid dates.']

log.loc[len(log)] = [
    'Date conversion: ' + 'Date Rptd',
    'Month persistence requires usable dates.',
    'Nonmissing input converted to NaT.',
    date_failures['Date Rptd'],
    'Preserve backup; flag failures; exclude unusable dates from dated analysis.',
    'Cannot recover invalid dates.']

In [45]:
log.loc[len(log)] = [
    'flag_occ_after_report',
    'Date order may affect the assessment period.',
    'Explicit screening predicate in item 8.',
    df['flag_occ_after_report'].sum(),
    'Flag and retain in master.',
    'Screening does not prove an error; extraction date and precise geography remain uncertain.']

log.loc[len(log)] = [
    'flag_future_date',
    'Future dates cannot support current assessment evidence.',
    'Explicit screening predicate in item 8.',
    df['flag_future_date'].sum(),
    'Flag and retain in master.',
    'Screening does not prove an error; extraction date and precise geography remain uncertain.']

log.loc[len(log)] = [
    'flag_very_old_occurrence',
    'Historical events may not reflect current field priorities.',
    'Explicit screening predicate in item 8.',
    df['flag_very_old_occurrence'].sum(),
    'Flag and retain in master.',
    'Screening does not prove an error; extraction date and precise geography remain uncertain.']

In [46]:
log.loc[len(log)] = [
    'flag_pre2024',
    'Legacy-period comparability requires review.',
    'Explicit screening predicate in item 8.',
    df['flag_pre2024'].sum(),
    'Flag and retain in master.',
    'Screening does not prove an error; extraction date and precise geography remain uncertain.']

log.loc[len(log)] = [
    'flag_transition_2024',
    'System-transition records need separate interpretation.',
    'Explicit screening predicate in item 8.',
    df['flag_transition_2024'].sum(),
    'Flag and retain in master.',
    'Screening does not prove an error; extraction date and precise geography remain uncertain.']

log.loc[len(log)] = [
    'flag_missing_coordinates',
    'Spatial matching cannot use missing coordinates.',
    'Explicit screening predicate in item 8.',
    df['flag_missing_coordinates'].sum(),
    'Flag and retain in master.',
    'Screening does not prove an error; extraction date and precise geography remain uncertain.']

In [47]:
log.loc[len(log)] = [
    'flag_invalid_coordinates',
    'Non-numeric coordinates cannot locate an assessment.',
    'Explicit screening predicate in item 8.',
    df['flag_invalid_coordinates'].sum(),
    'Flag and retain in master.',
    'Screening does not prove an error; extraction date and precise geography remain uncertain.']

log.loc[len(log)] = [
    'flag_zero_coordinates',
    'Zero coordinates are suspicious for Los Angeles.',
    'Explicit screening predicate in item 8.',
    df['flag_zero_coordinates'].sum(),
    'Flag and retain in master.',
    'Screening does not prove an error; extraction date and precise geography remain uncertain.']

log.loc[len(log)] = [
    'flag_outside_la_box',
    'Outlying coordinates need review before spatial assessment.',
    'Explicit screening predicate in item 8.',
    df['flag_outside_la_box'].sum(),
    'Flag and retain in master.',
    'Screening does not prove an error; extraction date and precise geography remain uncertain.']

In [48]:
log.loc[len(log)] = [
    'HHMM validation',
    'Time classification determines the nighttime assessment pool.',
    'Digits, hours 0–23, minutes 0–59.',
    df['flag_invalid_time'].sum(),
    'Preserve source; flag invalid; exclude from denominator.',
    'Invalid time cannot be inferred.']

log.loc[len(log)] = [
    'Missing times',
    'Keep the nighttime denominator transparent.',
    'Missing source-time flag.',
    df['flag_missing_time'].sum(),
    'Retain; exclude from denominator.',
    'Missing-time patterns may bias geographic comparisons.']

log.loc[len(log)] = [
    'Ambiguous 0000',
    'Show uncertainty in which districts receive attention.',
    'Source inspection and independent flag check.',
    df['flag_0000_ambiguous'].sum(),
    'Preserve and flag; exclude from main percentage; show inclusive count.',
    'Midnight versus unknown precision is unresolved.']

In [49]:
log.loc[len(log)] = [
    '2025-forward focus',
    'Keep the initial screen relevant while retaining historical evidence.',
    'Occurrence >= 2025-01-01.',
    (~df['analysis_post2024']).sum(),
    'Flag only; no date-based master deletion.',
    'Historical and missing dates excluded from focus.']

log.loc[len(log)] = [
    'Nighttime 20:00–05:59',
    'Define a consistent initial assessment window.',
    'Compare valid hour to stated range.',
    df['nighttime_unambiguous'].sum(),
    'Create inclusive and unambiguous classifications.',
    'Fixed hours do not follow seasonal sunset or sunrise.']

log.loc[len(log)] = [
    'No streetlight merge',
    'Offense patterns alone cannot establish lighting quality.',
    'No join operations performed.',
    0,
    'Not applicable at this stage.',
    'Future joins require authorization and spatial validation.']

In [50]:
display(log)
log.to_csv(OUT / 'cleaning_decision_log.csv', index=False)

,rule,project_specific_reason,verification_check,affected_records,final_treatment,remaining_limitation
0,Preserve source and identifiers,Keep assessment priorities traceable to origin...,SHA-256 before and after; strings preserve lea...,489955.0,Source unchanged; work on a copy.,Live-portal completeness unverified.
1,Whitespace: AREA NAME,Avoid splitting equivalent location or offense...,Compare original and normalized strings.,489955.0,Trim/collapse whitespace only when changed; ba...,Wording and capitalization are not reconciled.
2,Whitespace: NIBR Description,Avoid splitting equivalent location or offense...,Compare original and normalized strings.,82058.0,Trim/collapse whitespace only when changed; ba...,Wording and capitalization are not reconciled.
3,Whitespace: Crime Against,Avoid splitting equivalent location or offense...,Compare original and normalized strings.,0.0,Trim/collapse whitespace only when changed; ba...,Wording and capitalization are not reconciled.
4,Whitespace: Premise Desc,Avoid splitting equivalent location or offense...,Compare original and normalized strings.,0.0,Trim/collapse whitespace only when changed; ba...,Wording and capitalization are not reconciled.
5,Whitespace: LOCATION,Avoid splitting equivalent location or offense...,Compare original and normalized strings.,0.0,Trim/collapse whitespace only when changed; ba...,Wording and capitalization are not reconciled.
6,Blank strings to missing,Do not invent premise or location evidence for...,Before/after missingness audit.,349949.0,Preserve missingness; add flags.,Missing information may vary by district.
7,Exact source duplicates,Avoid double-counting identical supplied offen...,All original columns match.,0.0,Keep first exact copy only.,Distinct rows with repeated IDs remain unresol...
8,Repeated case IDs,Multiple offenses in one case must not disappe...,Compare offense IDs and case IDs.,94224.0,Retain; report distinct cases separately.,Offense counts and case counts differ.
9,Repeated offense IDs,Avoid silently resolving conflicting offense e...,Identifier audit and sample rows.,0.0,Flag and retain.,Nonunique IDs can affect interpretation of off...


## 13. Test reproducibility and export the sample

The sample starts with the five verified records, includes up to two examples of each flag, and adds evenly spaced source rows until it reaches 50. Keeping the first selection reason gives priority to the checked/difficult cases. It is a purposive review sample, not a representative sample for estimating rates. The short loops only collect row labels and reasons; the analysis rules above remain explicit.

In [51]:
# Keep the same flag order used in the cleaning steps.
flag_columns = ['flag_missing_premise', 'flag_missing_location', 'flag_missing_identifier',
    'flag_repeated_UniqueNIBRNo', 'flag_repeated_CaseNo', 'flag_invalid_Date_OCC',
    'flag_invalid_Date_Rptd', 'flag_occ_after_report', 'flag_future_date',
    'flag_very_old_occurrence', 'flag_pre2024', 'flag_transition_2024',
    'flag_before_transition_2024', 'flag_valid_time', 'flag_missing_time',
    'flag_invalid_time', 'flag_0000_ambiguous', 'flag_missing_coordinates',
    'flag_invalid_coordinates', 'flag_zero_coordinates', 'flag_outside_la_box']

sample_choices = pd.DataFrame()
sample_choices['row_index'] = chosen
sample_choices['sample_selection_reason'] = checks['selection_reason']
for flag in flag_columns:
    difficult_rows = df.loc[df[flag]].head(2)
    choices = pd.DataFrame({'row_index': difficult_rows.index})
    choices['sample_selection_reason'] = flag
    sample_choices = pd.concat([sample_choices, choices], ignore_index=True)
sample_choices = sample_choices.drop_duplicates('row_index', keep='first').head(50).reset_index(drop=True)

In [52]:
number_of_positions = min(50, df.shape[0])
for position in range(number_of_positions):
    row_position = position * df.shape[0] // number_of_positions
    row_index = df.index[row_position]
    sample_choices.loc[len(sample_choices)] = [row_index, 'Deterministically spaced context row']

sample_choices = sample_choices.drop_duplicates('row_index', keep='first').head(50)
sample = df.loc[sample_choices['row_index']].copy()
sample['sample_selection_reason'] = sample_choices['sample_selection_reason'].to_numpy()
sample.to_csv(OUT / 'lapd_nibrs_cleaned_sample.csv', index=False)
print('Sample rows:', sample.shape[0])
sample[['source_row', 'UniqueNIBRNo', 'sample_selection_reason']]

Sample rows: 50


,source_row,UniqueNIBRNo,sample_selection_reason
2,4,26170796_13B_0,Typical valid nighttime
53,55,26172245_220_0,Ambiguous midnight
628,630,971726272__1,Missing premise
150,152,26179469_270_0,Historical occurrence
0,2,25088246_13A_0,Multiple offenses in a case
632,634,990518378__1,flag_missing_premise
1,3,25088246_290_2,flag_repeated_CaseNo
6766,6768,241713274__1,flag_occ_after_report
6767,6769,241713274__0,flag_occ_after_report
623,625,80055440_09A_0,flag_very_old_occurrence


### Plain checks after running from the original CSV

Each assertion has a message describing the failed expectation. The source fingerprint is recalculated to verify preservation. Only three CSVs are written; audits and district/month previews remain displayed above, with complete result tables in memory. The large cleaned master stays in memory as `df`.

In [53]:
assert starting_rows - duplicate_count == df.shape[0], 'Row reconciliation failed.'
assert rows_before_conversion == df.shape[0], 'Date conversion changed the row count.'
assert len(pd.Index(required).difference(df.columns)) == 0, 'A required column is missing.'

valid_times = df.loc[df['flag_valid_time'], 'time_hhmm']
assert valid_times.str.len().eq(4).all(), 'A valid time is not four characters.'
assert valid_times.str[:2].astype(int).between(0, 23).all(), 'A valid hour is outside 00–23.'
assert valid_times.str[2:].astype(int).between(0, 59).all(), 'A valid minute is outside 00–59.'
assert df.loc[df['flag_0000_ambiguous'], 'time_hhmm'].eq('0000').all(), 'An ambiguous zero time was lost.'

assert checks.shape[0] == 5, 'Five record checks are required.'
assert checks['source_row'].nunique() == 5, 'The five checks must use distinct rows.'
assert checks['match'].all(), 'At least one expected result does not match.'
assert sample.shape[0] <= 50, 'The review sample is too large.'

In [54]:
with RAW.open('rb') as source_file:
    ending_hash = hashlib.file_digest(source_file, 'sha256').hexdigest()
assert ending_hash == source_hash, 'The raw file changed.'
assert (OUT / 'lapd_nibrs_cleaned_sample.csv').exists(), 'The cleaned sample is missing.'
assert (OUT / 'cleaning_decision_log.csv').exists(), 'The decision log is missing.'
assert (OUT / 'five_record_checks.csv').exists(), 'The record checks are missing.'

print('Offense identifiers unique:', df['UniqueNIBRNo'].is_unique)
print('Missing offense identifiers:', df['UniqueNIBRNo'].isna().sum())
print('All checks passed. The raw source is unchanged.')
print('Only the sample, decision log, and five-record checks were exported.')

Offense identifiers unique: True
Missing offense identifiers: 0
All checks passed. The raw source is unchanged.
Only the sample, decision log, and five-record checks were exported.


## 14. Document limitations
- Reported offenses do not represent all offenses or community risk. Reporting and enforcement patterns can influence what enters this file.
- The March 7, 2024 records-system transition limits comparisons. A 2025-forward focus does not by itself prove stable or complete reporting coverage.
- `0000` may mean midnight or uncertain time precision. The conservative denominator excludes it; the inclusive count shows sensitivity to that choice.
- Hundred-block locations and coordinates are approximate. A broad bounding box cannot validate exact location or establish a city boundary.
- Streetlight asset locations do not measure illumination quality. Streetlight `STATUS` is not verified real-time operating condition without documentation.
- Offense counts do not measure individual risk without exposure denominators such as pedestrian volume, nighttime population, or roadway activity.
- Geographic overlap cannot establish that inadequate lighting caused an offense. On-site assessment and additional evidence are required before infrastructure recommendations.
- Counts of active months do not establish uninterrupted recurrence; review the monthly table. Partial months and unknown extraction time constrain persistence comparisons.
- The supplied export is far larger than 2,500 rows, but completeness against the live source has not been verified. The purposive cleaned sample is for review only.

### Reproducibility

The current notebook was restarted and run from the original CSV. All cells completed without error, the raw-file fingerprint remained unchanged, and the three submitted CSV outputs were recreated.